In [ ]:
import pandas as pd # for data manipulation
import numpy as np # for data manipulation
import scipy.optimize as opt # for optimization

### Exercise 01 - Regression
**Jenny Laberg Nilsson**  
**BERN02 - 2026/09**

**Project:** Fit a local regression over each point X0 =[10,18,25]  
**Quantity of interest**: Bird count  
**Predictors**: MORT - (Total age-adjusted mortality rate per 100,000) and POOR - (% of families with income < $3000)  
**Model**: Local regression    

In [2]:
# Load pollution data
pollution_df = pd.read_csv("../data/pollution_cleaneddata.csv", encoding='utf-8')
# print the first 5 rows of the dataframe
pollution_df.head()

,PREC,JANT,JULT,OVR65,POPN,EDUC,HOUS,DENS,NONW,WWDRK,POOR,HC,NOX,SO@,HUMID,MORT
0,36.0,27.0,71.0,8.1,3.34,11.4,81.5,3243.0,8.8,42.6,11.7,21.0,15.0,59.0,59.0,921.870
1,35.0,23.0,72.0,11.1,3.14,11.0,78.8,4281.0,3.5,50.7,14.4,8.0,10.0,39.0,57.0,997.875
2,44.0,29.0,74.0,10.4,3.21,9.8,81.6,4260.0,0.8,39.4,12.4,6.0,6.0,33.0,54.0,962.354
3,47.0,45.0,79.0,6.5,3.41,11.1,77.5,3125.0,27.1,50.2,20.6,18.0,8.0,24.0,56.0,982.291
4,43.0,35.0,77.0,7.6,3.44,9.6,84.6,6441.0,24.4,43.7,14.3,43.0,38.0,206.0,55.0,1071.289


In [ ]:
def local_x(x, y, x0, k):
    """
    Find the closest points to x0 by calculating the squared distance between each point in x and x0.
    k is the number of closest points to return also known as the bandwidth.
    The function returns a list of tuples containing the squared distance and the x and y coordinates for the k closest points.
    """
    # make a empty list for storing the squared distances between each point in x and x0 and the coordinates of the points in x and y
    squared_distances_list = []
    # loop through each point in x and calculate the squared distance between the point and x0
    for i in range(len(x)):
        squared_distance = float((x[i] - x0) ** 2)
        y_value = float(y[i])
        x_value = float(x[i])
        squared_distances_list.append((squared_distance, x_value, y_value))
    # sort list by the squared distances
    squared_distances_list.sort()
    # return the k closest points
    return squared_distances_list[:k]


In [4]:
# Just trying it out but I should definitely think about how to set k
print(local_x(pollution_df['POOR'], pollution_df['MORT'], 10, 5))

[(0.009999999999999929, 10.1, 899.264), (0.039999999999999716, 9.8, 857.622), (0.09000000000000043, 9.7, 790.733), (0.09000000000000043, 10.3, 967.803), (0.25, 9.5, 887.466)]


In [ ]:
def weight_function(x, x0, k):
    """
    Calculate the weights for each point in x based on the distance from x0 using a Gaussian kernel (function found online).
    """
    # make an empty list for stroing the weights
    weights = []
    # loop through each point in x and calculate the weight based on the distance from x0
    for i in range(len(x)):
        # calculate the weight using a Gaussian kernel
        weight = np.exp(-((x[i] - x0) ** 2) / (2 * (k ** 2)))
        weights.append(weight)
    return weights

In [ ]:
def optimize_sum_of_squares(beta, x_closest, y_closest, weights):
    """
    Find the optimal beta parameters for the weighted linear regression model by minimizing the sum of squares.
    """
    beta_0 = beta[0]
    beta_1 = beta[1]

    # calculate the sum of squares
    eq = weights * (y_closest - (beta_0 + beta_1 * x_closest)) ** 2
    return(eq.sum())

In [22]:
def local_regression(x, y, x0, k):
    """
    Perform local regression on the data using the k closest points to x0. 
    Returns the predicted value of y at x0 and the standard deviation of the residuals.
    """
    # If x0 is a list or array, run recursively for each item
    if isinstance(x0, (list, np.ndarray)):
        return [local_regression(x, y, val, k) for val in x0]
        
    # get the closets x and their coordinates of the k closest points to x0 by calling the local_x function
    closest_points = local_x(x, y, x0, k)

    # extract only the coordinates and make it into an array for easier calculations
    x_closest = np.array([point[1] for point in closest_points])
    y_closest = np.array([point[2] for point in closest_points])

    # I also have to calculate the weights for each point in x_closest based on the distance from x0 by calling the weight_function
    weights = weight_function(x_closest, x0, k)

    # find the most optimal beta parametes (fit the model)
    beta = opt.minimize(optimize_sum_of_squares, [0, 0], args=(x_closest, y_closest, weights)).x
    beta_0 = beta[0]
    beta_1 = beta[1]

    # predict y_0  
    y_0 = beta_0 + beta_1 * x0

    # standard deviation of the residuals
    residuals = y_closest - (beta[0] + beta[1] * x_closest)
    std_dev = np.sqrt(np.sum(weights * residuals ** 2))

    return float(y_0), float(std_dev)

In [21]:
X0 =[10,18,25]
test = local_regression(pollution_df['POOR'], pollution_df['MORT'], X0, 5)
print(test)

[(897.0645891930056, 94.09200859107074), (971.3445841338938, 90.00909150043424), (1024.996243726423, 109.12453793153163)]
